# 05 — Data cleansing: `quien_es_quien.csv` (esquema sucio, sin encabezados)

**Objetivo:** ingeniería de datos sobre un archivo real mal formateado — parseo,
inferencia de columnas, limpieza de nulos no estándar (`\N`), y estandarización de tipos.

**Dataset:** `quien_es_quien.csv` (~15.4 GB) — viendo una muestra real del archivo:

```
+--------------------+--------------------+--------+----------------+----------------+----+---+------------------+----------+--------------------+--------------------+----------------+--------------------+--------+---------+
|                 _c0|                 _c1|     _c2|             _c3|             _c4| _c5|_c6|               _c7|       _c8|                 _c9|                _c10|            _c11|                _c12|    _c13|     _c14|
+--------------------+--------------------+--------+----------------+----------------+----+---+------------------+----------+--------------------+--------------------+----------------+--------------------+--------+---------+
|CUADERNO FORMA IT...|96 HOJAS PASTA DU...|ESTRELLA|MATERIAL ESCOLAR|UTILES ESCOLARES|25.9| \N|ABASTECEDORA LUMEN|PAPELERIAS|ABASTECEDORA LUME...|CANNES No. 6 ESQ....|DISTRITO FEDERAL|TLALPAN          ...|19.29699|-99.12542|
```

Spark no encontró encabezados (por eso los nombres genéricos `_c0`..`_c14`) y `\N` es el
marcador de nulo de MySQL — no lo reconoce como `NULL` automáticamente, hay que decirle.

**Dato clave:** son **15 columnas** (`_c0` a `_c14`), el mismo número y el mismo orden de
columnas que `all_data.csv` (que sí trae encabezados) — ver `recursos/datasets/README.md`,
Sección 1. Todo indica que es el mismo dataset (PROFECO, "Quién es Quién en los Precios"),
solo que exportado sin encabezados y sin limpiar. Eso convierte este notebook en un
ejercicio real: reconstruir el esquema comparando contra la versión limpia.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType

spark = SparkSession.builder.appName("05_data_cleansing").getOrCreate()

RUTA = "gs://<TU-BUCKET>/raw/profeco/quien_es_quien.csv"

## 1. Leer sin encabezados y confirmar la hipótesis del esquema

In [ ]:
df_crudo = spark.read.csv(RUTA, header=False, inferSchema=True)
df_crudo.printSchema()
df_crudo.show(2, truncate=True)

## 2. Asignar nombres de columna (inferidos de `all_data.csv`)

Mismo orden de columnas que la versión con encabezados del dataset.

In [ ]:
COLUMNAS = [
    "producto", "presentacion", "marca", "categoria", "catalogo",
    "precio", "fechaRegistro", "cadenaComercial", "giro", "nombreComercial",
    "direccion", "estado", "municipio", "latitud", "longitud",
]

df_nombrado = df_crudo.toDF(*COLUMNAS)
df_nombrado.printSchema()

## 3. Limpiar el marcador de nulo `\N` y castear tipos

`\N` llega como el string literal `"\\N"` (no como nulo real) porque `inferSchema` no
lo reconoce -- toca reemplazarlo explícitamente antes de castear, o el cast silenciosamente
produce `null` de todos modos pero sin que quede registrado *por qué* quedó nulo.

In [ ]:
def limpiar_nulos_mysql(df, columnas):
    for col in columnas:
        df = df.withColumn(col, F.when(F.col(col) == "\\N", None).otherwise(F.col(col)))
    return df

df_limpio = limpiar_nulos_mysql(df_nombrado, COLUMNAS)

df_limpio = (
    df_limpio
    .withColumn("precio", F.col("precio").cast(DoubleType()))
    .withColumn("latitud", F.col("latitud").cast(DoubleType()))
    .withColumn("longitud", F.col("longitud").cast(DoubleType()))
)

## 4. Estandarizar texto

Varios valores llegan con espacios de relleno (padding) al final -- por ejemplo
`"TLALPAN          "` en vez de `"TLALPAN"`. Se ve en el `show()` de la Sección 1
aunque quede truncado en la vista previa.

In [ ]:
columnas_texto = ["producto", "presentacion", "marca", "categoria", "catalogo",
                   "cadenaComercial", "giro", "nombreComercial", "direccion",
                   "estado", "municipio"]

for col in columnas_texto:
    df_limpio = df_limpio.withColumn(col, F.trim(F.upper(F.col(col))))

## 5. Validación de calidad antes de dar por buena la limpieza

In [ ]:
def validar_calidad(df):
    errores = []

    nulos_precio = df.filter(F.col("precio").isNull()).count()
    if nulos_precio > 0:
        errores.append(f"{nulos_precio} filas con precio nulo tras el cast.")

    coords_invalidas = df.filter(
        (F.col("latitud") < -90) | (F.col("latitud") > 90) |
        (F.col("longitud") < -180) | (F.col("longitud") > 180)
    ).count()
    if coords_invalidas > 0:
        errores.append(f"{coords_invalidas} filas con coordenadas fuera de rango.")

    if errores:
        print("Validación de calidad -- avisos (no se detiene el pipeline, se reporta):")
        for e in errores:
            print(f"  - {e}")
    else:
        print("Validación de calidad: OK")


validar_calidad(df_limpio)
df_limpio.show(5, truncate=False)

## 6. Guardar como capa "silver" (limpia, tipada, sin nulos-fantasma)

Preparación directa para la Sesión 6 (Data Lakes / arquitectura medallion).

In [ ]:
df_limpio.write.mode("overwrite").partitionBy("estado").parquet(
    "gs://<TU-BUCKET>/processed/quien_es_quien_limpio/"
)

In [ ]:
spark.stop()